# 12 — Operability Demo Handoff

## Purpose

This notebook continues after `11_realtime_endpoint_validation.ipynb` and packages the AeroDelay ML system into a stakeholder-ready handoff. It validates that the major system assets exist, creates a machine-readable system manifest, and writes a concise run-of-show that the team can use for a professional screencast under 10 minutes.

## Prerequisites

Run notebooks `01_s3_setup.ipynb` through `10_model_registry.ipynb` first. Run notebook `11_realtime_endpoint_validation.ipynb` if the team wants to include real-time endpoint proof in the final screencast. This notebook is read-only except for writing report artifacts to the local `reports/` folder and S3.


## 1. Setup

This step recreates the project clients and prepares a local report folder. It follows the same setup pattern used in notebooks 07 through 11.


In [ ]:
import boto3, os, json, textwrap
import pandas as pd
import awswrangler as wr
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket = sess.default_bucket()
role = get_execution_role()
region = sess.boto_region_name

sm = boto3.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)

os.makedirs("reports", exist_ok=True)

print(f"Bucket : {bucket}")
print(f"Region : {region}")


In [ ]:
%store -r s3_aerodelay
%store -r baseline_job_name
%store -r baseline_model_uri
%store -r best_model_uri
%store -r best_hpo_job_name
%store -r model_package_group
%store -r model_package_arn
%store -r batch_job_name
%store -r batch_output_s3

try:
    %store -r endpoint_name
except Exception:
    endpoint_name = "aerodelay-xgb-realtime-endpoint"

try:
    %store -r realtime_validation_s3
except Exception:
    realtime_validation_s3 = f"{s3_aerodelay}/reports/realtime_endpoint_validation.json"

print(f"s3_aerodelay          : {s3_aerodelay}")
print(f"best_hpo_job_name     : {best_hpo_job_name}")
print(f"model_package_group   : {model_package_group}")
print(f"batch_output_s3       : {batch_output_s3}")
print(f"endpoint_name         : {endpoint_name}")


## 2. Validate End-to-End ML System Assets

This section checks whether each major artifact from the modular workflow is available. The goal is to give the team a single operational checklist that can be shown during the screencast.


In [ ]:
def check_s3_uri(name, s3_uri):
    bucket_name = s3_uri.split("/")[2]
    key = "/".join(s3_uri.split("/")[3:])
    try:
        response = s3.head_object(Bucket=bucket_name, Key=key)
        return {
            "component": name,
            "status": "available",
            "location": s3_uri,
            "detail": f"{response['ContentLength']/1024:.1f} KB",
        }
    except Exception as e:
        return {
            "component": name,
            "status": "missing_or_unavailable",
            "location": s3_uri,
            "detail": str(e),
        }

checks = []
checks.append(check_s3_uri("Training split", f"{s3_aerodelay}/training/data.parquet"))
checks.append(check_s3_uri("Validation split", f"{s3_aerodelay}/validation/data.parquet"))
checks.append(check_s3_uri("Test split", f"{s3_aerodelay}/test/data.parquet"))
checks.append(check_s3_uri("Production simulation split", f"{s3_aerodelay}/production_simulation/data.parquet"))
checks.append(check_s3_uri("Baseline model artifact", baseline_model_uri))
checks.append(check_s3_uri("Selected XGBoost model artifact", best_model_uri))
checks.append(check_s3_uri("Evaluation report", f"{s3_aerodelay}/reports/evaluation_report.json"))
checks.append(check_s3_uri("Batch predictions", f"{s3_aerodelay}/batch-output/production_features.csv.out"))
checks.append(check_s3_uri("Real-time endpoint validation", realtime_validation_s3))

try:
    group_desc = sm.describe_model_package_group(ModelPackageGroupName=model_package_group)
    checks.append({
        "component": "Model package group",
        "status": group_desc["ModelPackageGroupStatus"],
        "location": model_package_group,
        "detail": "SageMaker Model Registry",
    })
except Exception as e:
    checks.append({
        "component": "Model package group",
        "status": "missing_or_unavailable",
        "location": model_package_group,
        "detail": str(e),
    })

try:
    endpoint_desc = sm.describe_endpoint(EndpointName=endpoint_name)
    checks.append({
        "component": "Real-time endpoint",
        "status": endpoint_desc["EndpointStatus"],
        "location": endpoint_name,
        "detail": "SageMaker endpoint",
    })
except Exception as e:
    checks.append({
        "component": "Real-time endpoint",
        "status": "not_deployed_or_deleted",
        "location": endpoint_name,
        "detail": str(e),
    })

system_checks_df = pd.DataFrame(checks)
display(system_checks_df)

system_checks_df.to_csv("reports/system_operability_checks.csv", index=False)
wr.s3.to_csv(system_checks_df, f"{s3_aerodelay}/reports/system_operability_checks.csv", index=False)
print("Saved system operability checks locally and to S3.")


## 3. Load Final Evaluation and Batch-Inference Summary

This step collects the quantitative proof points that should appear in the stakeholder demonstration: model selection, test-set metrics, and production-simulation batch output.


In [ ]:
evaluation_report_path = f"{s3_aerodelay}/reports/evaluation_report.json"

try:
    eval_report_df = wr.s3.read_json(evaluation_report_path)
    selected_model = eval_report_df["selected_model"].iloc[0]
    xgb_metrics = eval_report_df["xgboost_best_hpo"].iloc[0]
    baseline_metrics = eval_report_df["logistic_regression"].iloc[0]
except Exception as e:
    print(f"Could not load evaluation report: {e}")
    selected_model = "xgboost_best_hpo"
    xgb_metrics = {}
    baseline_metrics = {}

try:
    pred_df = wr.s3.read_csv(
        path=f"{s3_aerodelay}/batch-output/production_features.csv.out",
        header=None,
        names=["delay_probability"],
    )
    batch_summary = {
        "total_predictions": int(len(pred_df)),
        "predicted_delay_rate": float((pred_df["delay_probability"] >= 0.5).mean()),
        "mean_delay_probability": float(pred_df["delay_probability"].mean()),
        "min_delay_probability": float(pred_df["delay_probability"].min()),
        "max_delay_probability": float(pred_df["delay_probability"].max()),
    }
except Exception as e:
    print(f"Could not load batch output: {e}")
    batch_summary = {}

print("Selected model:", selected_model)
print("XGBoost metrics:", xgb_metrics)
print("Baseline metrics:", baseline_metrics)
print("Batch summary:", batch_summary)


## 4. Create System Manifest

The manifest is a compact handoff artifact that summarizes the project resources, model lineage, evaluation evidence, and inference assets. It is useful for the final report, GitHub documentation, and the stakeholder screencast.


In [ ]:
system_manifest = {
    "project": "AeroDelay AI",
    "business_problem": "Predict arrival delays of 15 minutes or more so airline stakeholders can anticipate operational risk before arrival.",
    "region": region,
    "s3_project_prefix": s3_aerodelay,
    "selected_model": selected_model,
    "baseline_model_uri": baseline_model_uri,
    "best_model_uri": best_model_uri,
    "best_hpo_job_name": best_hpo_job_name,
    "model_package_group": model_package_group,
    "model_package_arn": model_package_arn,
    "batch_job_name": batch_job_name,
    "batch_output_s3": batch_output_s3,
    "endpoint_name": endpoint_name,
    "realtime_validation_s3": realtime_validation_s3,
    "xgboost_metrics": xgb_metrics,
    "baseline_metrics": baseline_metrics,
    "batch_summary": batch_summary,
    "google_colab_implementation": "Reference implementation prepared for the modular notebook workflow.",
    "gradio_demo_link": "https://0c6f8a94fc0bf5f4b2.gradio.live",
}

with open("reports/aerodelay_system_manifest.json", "w") as f:
    json.dump(system_manifest, f, indent=2, default=str)

wr.s3.to_json(
    df=pd.DataFrame([system_manifest]),
    path=f"{s3_aerodelay}/reports/aerodelay_system_manifest.json",
)

system_manifest_s3 = f"{s3_aerodelay}/reports/aerodelay_system_manifest.json"
%store system_manifest_s3
print(f"System manifest saved locally and to: {system_manifest_s3}")


## 5. Build the Under-10-Minute Demonstration Run-of-Show

This section creates a practical outline for the team presentation. It keeps the demonstration stakeholder-focused while still proving the required ML system components.


In [ ]:
demo_rows = [
    {
        "segment": "1. Business problem and value",
        "time_box": "0:00-1:00",
        "owner": "Presenter 1",
        "proof_point": "Flight delays create operational and customer-service risk; AeroDelay predicts arrival delay risk before arrival.",
        "visual_or_notebook": "Slide title + notebook 03 EDA delay-rate evidence",
    },
    {
        "segment": "2. Data and feature pipeline",
        "time_box": "1:00-2:20",
        "owner": "Presenter 1",
        "proof_point": "S3, Athena, preprocessing, engineered rates, and Feature Store support reproducible training data.",
        "visual_or_notebook": "Notebooks 01-06",
    },
    {
        "segment": "3. Model development",
        "time_box": "2:20-4:00",
        "owner": "Presenter 2",
        "proof_point": "Logistic Regression creates a defensible baseline; XGBoost with HPO improves the selected operating metrics.",
        "visual_or_notebook": "Notebooks 07-08",
    },
    {
        "segment": "4. Evaluation and model selection",
        "time_box": "4:00-5:30",
        "owner": "Presenter 2",
        "proof_point": "Evaluation compares baseline and XGBoost using recall, F1, ROC-AUC, confusion matrices, and subgroup analysis.",
        "visual_or_notebook": "Notebook 09 + reports figures",
    },
    {
        "segment": "5. Registry and batch inference",
        "time_box": "5:30-7:00",
        "owner": "Presenter 3",
        "proof_point": "The selected model is approved in Model Registry and produces batch predictions on production-simulation data.",
        "visual_or_notebook": "Notebook 10",
    },
    {
        "segment": "6. Real-time operability",
        "time_box": "7:00-8:15",
        "owner": "Presenter 3",
        "proof_point": "Endpoint validation demonstrates that the system can score a single flight record for an operational use case.",
        "visual_or_notebook": "Notebook 11",
    },
    {
        "segment": "7. Handoff, monitoring, and next steps",
        "time_box": "8:15-9:45",
        "owner": "All presenters",
        "proof_point": "System manifest, operability checks, monitor baseline, cleanup controls, and demo link make the solution defensible and presentable.",
        "visual_or_notebook": "Notebook 12 + Gradio demo link",
    },
]

demo_run_of_show_df = pd.DataFrame(demo_rows)
display(demo_run_of_show_df)

demo_run_of_show_df.to_csv("reports/demo_run_of_show.csv", index=False)
wr.s3.to_csv(demo_run_of_show_df, f"{s3_aerodelay}/reports/demo_run_of_show.csv", index=False)
print("Saved demo run-of-show locally and to S3.")


## 6. Write Stakeholder Transcript Draft

This cell writes a concise transcript that the team can edit before recording. The transcript is intentionally business-facing and avoids turning the demonstration into a code walkthrough.


In [ ]:
transcript = f"""
# AeroDelay AI — Stakeholder Demonstration Transcript Draft

This demonstration presents AeroDelay AI as an operational machine learning solution for predicting whether a flight is likely to arrive at least 15 minutes late. The business goal is not simply to build a high-scoring model; it is to give airline stakeholders earlier visibility into delay risk so they can plan staffing, gates, customer communications, and recovery actions.

The system starts with raw Bureau of Transportation Statistics flight records stored in S3. Athena supports repeatable querying, and the preprocessing workflow creates clean train, validation, test, and production-simulation splits. Feature engineering adds operational signals such as route, carrier, origin, destination, and hour-level historical delay rates. Those features are stored and validated so downstream training uses a consistent input definition.

For model development, the team trained a Logistic Regression baseline and then an XGBoost classifier with hyperparameter tuning. The selected model is `{selected_model}` because it provides the strongest overall operating profile on the held-out test set, especially for delay-oriented metrics such as recall, F1, and ROC-AUC. The evaluation notebook includes confusion matrices, ROC and precision-recall curves, feature importance, and subgroup checks so stakeholders can understand both performance and limitations.

The selected XGBoost model is registered in SageMaker Model Registry under `{model_package_group}`. Batch Transform then scores production-simulation records and writes predictions to `{batch_output_s3}`. For operational validation, the real-time endpoint workflow can score a single flight record and return a delay probability for dashboard or decision-support use cases.

The final handoff includes system checks, the manifest file, endpoint validation output, model registry evidence, and a demo-ready Gradio interface at https://0c6f8a94fc0bf5f4b2.gradio.live. The recommended next steps are to connect the model to live schedule feeds, monitor feature drift and prediction quality, review subgroup performance over time, and define a controlled retraining cadence.
""".strip()

with open("reports/stakeholder_demo_transcript.md", "w") as f:
    f.write(transcript + "\n")

s3.put_object(
    Bucket=bucket,
    Key="airline-delay/reports/stakeholder_demo_transcript.md",
    Body=transcript.encode("utf-8"),
    ContentType="text/markdown",
)

stakeholder_transcript_s3 = f"{s3_aerodelay}/reports/stakeholder_demo_transcript.md"
%store stakeholder_transcript_s3
print(transcript[:1000])
print(f"\nTranscript saved locally and to: {stakeholder_transcript_s3}")


## 7. Final Commit Checklist

This final cell creates a small checklist that explains what should be committed and how the team should present notebooks 11 and 12 separately.


In [ ]:
checklist = """
# Notebook 11 and 12 Commit Checklist

## Commit 1 — Real-Time Endpoint Validation

Commit `11_realtime_endpoint_validation.ipynb` separately. This notebook should be described as the real-time inference validation module that follows model registry and batch inference.

Suggested commit message:

`Add notebook 11 for SageMaker real-time endpoint validation`

## Commit 2 — Operability Demo Handoff

Commit `12_operability_demo_handoff.ipynb` separately. This notebook should be described as the stakeholder demo handoff module that validates system assets and writes the run-of-show, transcript, and manifest.

Suggested commit message:

`Add notebook 12 for ML system operability handoff`

## Before Recording

Confirm that the final video is under 10 minutes, each presenter speaks for a comparable amount of time, notebook execution outputs are clean, endpoint cleanup is shown or discussed, and glitches/outtakes are edited out.
""".strip()

with open("reports/notebook_11_12_commit_checklist.md", "w") as f:
    f.write(checklist + "\n")

s3.put_object(
    Bucket=bucket,
    Key="airline-delay/reports/notebook_11_12_commit_checklist.md",
    Body=checklist.encode("utf-8"),
    ContentType="text/markdown",
)

print(checklist)


## 8. Summary — Operability Demo Handoff

This notebook completes the modular sequence by turning the completed ML workflow into a professional stakeholder handoff package. It does not retrain models or recreate infrastructure. Instead, it verifies the assets produced by notebooks 01 through 11 and writes the deliverables needed for the final screencast.

| Deliverable | Status | Location |
|---|---|---|
| System operability checks | Generated | `reports/system_operability_checks.csv` |
| System manifest | Generated | `reports/aerodelay_system_manifest.json` |
| Demo run-of-show | Generated | `reports/demo_run_of_show.csv` |
| Stakeholder transcript | Generated | `reports/stakeholder_demo_transcript.md` |
| Commit checklist | Generated | `reports/notebook_11_12_commit_checklist.md` |
| Gradio demo reference | Included | `https://0c6f8a94fc0bf5f4b2.gradio.live` |

The result is a defensible, modular handoff that demonstrates data readiness, model training, evaluation, registry governance, batch inference, real-time validation, monitoring readiness, and stakeholder communication.
